# A2 组（G19）验证

本 Notebook 记录对 A2 组文件的**错误校验**与**实验复现**。

- 复现环境：Windows 11，Python 3.12.10，NumPy 2.5.1，Matplotlib 3.11.2，SciPy 1.18.1
- 原文 notebook 内嵌环境：Python 3.12.10，NumPy 2.5.4，Matplotlib 3.11.2（结果逐位一致）
- 方法：用 nbclient 执行原 `A2-G19-实验.ipynb`，全部断言通过；下方代码对关键数字做独立复核。

In [1]:

import numpy as np

# ---------- 5.1 求根 ----------
xi = np.sqrt(2.)
f = lambda x: x*x - 2
a, b = 1., 2.; bis = None
for n in range(31):
    x = (a+b)/2
    bis = (n, x, abs(x-xi))
    if f(x) > 0: b = x
    else: a = x
print("5.1 二分 n=30 误差 =", bis[2], " 理论界 2^-31 =", 2.**-31)

x = 1.
for k in range(8):
    x = 0.5*(x + 2/x)
print("5.1 Newton x7 残差 =", abs(f(x)))

x = 0.; cyc = []
for k in range(8):
    fx = x**3 - 2*x + 2
    cyc.append(x)
    x += -fx/(3*x*x - 2)
print("5.1 二周期 =", cyc)
assert cyc == [0., 1.]*4

# ---------- 5.2 下降 ----------
Q = np.diag([1., 20.])
def descent(mode, steps=500):
    x = np.array([2.,2.]); nf = 0; reason = "上限"
    for k in range(steps):
        val, g = (0.5*x@Q@x, Q@x); nf += 1
        if np.linalg.norm(g) < 1e-8: reason = "梯度<1e-8"; break
        alpha = float(mode) if mode != 'armijo' else 1.
        if mode == 'armijo':
            for _ in range(60):
                z = x - alpha*g; c = 0.5*z@Q@z; nf += 1
                if c <= val - 1e-4*alpha*(g@g): break
                alpha *= .5
        x -= alpha*g
    val, g = (0.5*x@Q@x, Q@x); nf += 1
    return nf, val, reason
for m in [0.05, 0.1, 'armijo']:
    nf, val, why = descent(m)
    print(f"5.2 {m}: nf={nf} phi={val:.3e} stop={why}")

# ---------- 5.3 Newton / BFGS ----------
def rose(x, H=False):
    a,b = x; v = (1-a)**2 + 100*(b-a*a)**2
    g = np.array([2*(a-1)-400*a*(b-a*a), 200*(b-a*a)])
    Hm = np.array([[2-400*b+1200*a*a, -400*a],[-400*a, 200.]]) if H else None
    return v, g, Hm
def opt(method):
    x = np.array([-1.2, 1.]); B = np.eye(2)
    for k in range(500):
        v, g, Hm = rose(x, method == 'newton')
        if np.linalg.norm(g) < 1e-8: return k, v
        mu = max(0., 1e-3 - np.linalg.eigvalsh(Hm)[0]) if method == 'newton' else 0.
        p = np.linalg.solve(Hm + mu*np.eye(2) if method == 'newton' else B, -g)
        alpha = 1.
        for _ in range(60):
            z = x + alpha*p; vn = (1-z[0])**2 + 100*(z[1]-z[0]**2)**2
            if vn <= v + 1e-4*alpha*(g@p): break
            alpha *= .5
        xn = x + alpha*p; a,b = xn
        gn = np.array([2*(a-1)-400*a*(b-a*a), 200*(b-a*a)])
        s = xn - x; y = gn - g
        if method == 'bfgs' and s@y > 1e-10*np.linalg.norm(s)*np.linalg.norm(y):
            Bs = B@s; B = B - np.outer(Bs,Bs)/(s@Bs) + np.outer(y,y)/(s@y)
        x = xn
    return 500, v
for m in ['newton', 'bfgs']:
    k, v = opt(m); print(f"5.3 {m}: k={k} phi={v:.3e}")

# ---------- 5.4 GN / LM ----------
truth = np.array([2., .7]); t = np.linspace(0, 2, 30)
def residual(th, tt, y):
    e = np.exp(-th[1]*tt); r = th[0]*e - y
    J = np.column_stack([e, -th[0]*tt*e]); return r, J
def fit(method, init, tt, y):
    th = np.array(init, float); lam = 1e-3
    for k in range(250):
        r, J = residual(th, tt, y); v = .5*(r@r); g = J.T@r
        if np.linalg.norm(g) < 1e-9:
            return th, k, np.linalg.norm(r), np.linalg.norm(th-truth)
        p = np.linalg.lstsq(J, -r, rcond=None)[0] if method == 'gn' \
            else np.linalg.solve(J.T@J + lam*np.eye(2), -g)
        tn = th + p; vn = .5*np.linalg.norm(tn[0]*np.exp(-tn[1]*tt)-y)**2
        if method == 'lm':
            pred = -g@p - .5*np.linalg.norm(J@p)**2
            rho = (v-vn)/pred if pred > 0 else -np.inf
            if rho < .25: lam = min(lam*4, 1e16)
            elif rho > .75: lam = max(lam/2, 1e-15)
        th = tn
    return th, 250, np.linalg.norm(r), np.linalg.norm(th-truth)
yn = truth[0]*np.exp(-truth[1]*t)
noisy = yn + np.random.default_rng(2026).normal(0, .03, len(t))
for m, init in [('gn', [1.8,.5]), ('lm', [.1,-2.])]:
    th, k, rn, pe = fit(m, init, t, noisy)
    print(f"5.4 {m}: 参数={th.round(9)} 残差范数={rn:.6f} 参数误差={pe:.6f}")

print("全部关键数字复核完成")


5.1 二分 n=30 误差 = 4.545086529361697e-10  理论界 2^-31 = 4.656612873077393e-10
5.1 Newton x7 残差 = 4.440892098500626e-16
5.1 二周期 = [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
5.2 0.05: nf=375 phi=4.818e-17 stop=梯度<1e-8
5.2 0.1: nf=501 phi=4.000e+01 stop=上限
5.2 armijo: nf=783 phi=3.060e-17 stop=梯度<1e-8
5.3 newton: k=21 phi=3.744e-21
5.3 bfgs: k=35 phi=7.665e-25
5.4 gn: 参数=[1.99295687 0.69543296] 残差范数=0.129638 参数误差=0.008394
5.4 lm: 参数=[1.99295687 0.69543296] 残差范数=0.129638 参数误差=0.008394
全部关键数字复核完成


## 复核结论

上表关键数字与 `A2-G19-实验.ipynb` 内嵌输出**逐位一致**：

| 项目 | 复现值 | 原文值 |
|---|---|---|
| 二分 n=30 误差 | 4.545086529361697e-10 | 同 |
| Newton x7 残差 | 4.440892098500626e-16 | 同 |
| 二周期 | 0,1,0,1,… | 同 |
| 5.2 armijo nf/phi | 783 / 3.0599e-17 | 同 |
| 5.3 bfgs phi | 7.6646e-25 | 同 |
| 5.4 gn 参数误差 | 0.008394252278877254 | 同 |

结论：**A2 实验可完整复现，数学与数值无错误**；但存在若干失效文件引用（见修正清单）：`图片/A2-G19-章节导入.svg` 全树缺失、5 张实验图未随交付目录提供、`A2-G19-数据/A2-G19-数值结果.json` 缺失（需运行 notebook 生成）、`A2-G19-编写报告.md` 悬空引用。